# Reading, Parsing, and Flattening Nested JSON

In real-world data pipelines, JSON is everywhere:

- APIs (weather, CRM, logs, social media)
- NoSQL exports (MongoDB, Firestore)
- Nested data from streaming or CDC feeds

JSON gives flexibility but makes processing harder.
As a data engineer, you must know how to flatten nested fields so they fit into tabular formats (SQL tables, DataFrames, Parquet, Delta).

### 1. What Is JSON?

JSON (JavaScript Object Notation) is a text format for structured data.
 It represents data as key–value pairs, lists, and nested objects.

In [0]:
{
  "id": 1,
  "name": "John",
  "address": {
    "city": "Mumbai",
    "zip": "400001"
  },
  "skills": ["Python", "SQL", "Databricks"]
}

Nested objects (address) and arrays (skills) are common - and must be flattened for analytics.

## 2. Reading JSON Files in Python

In [0]:
import json

with open("/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_json1.json", "r") as f:
    data = json.load(f)

print(data)

b) Using pandas directly:

In [0]:
import pandas as pd

# Read the JSON file
df = pd.read_json(
    "/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_json1.json",
    typ="series"
)

# Convert Series → DataFrame
df = df.to_frame().T

print(df)


## 3. Flattening Nested JSON - Single-Level Nesting
We can flatten address.city and address.zip into columns.

In [0]:
import pandas as pd
from pandas import json_normalize

data = {
  "id": 1,
  "name": "Akash",
  "address": {"city": "Mumbai", "zip": "400001"},
  "skills": ["Python", "SQL", "Databricks"]
}

df = json_normalize(data)
print(df)

## 4. Flattening Nested Lists (One-to-Many Relationships)

In [0]:
data = [
  {
    "id": 1,
    "name": "Akash",
    "skills": ["Python", "SQL", "Databricks"]
  },
  {
    "id": 2,
    "name": "Priya",
    "skills": ["Excel", "PowerBI"]
  }
]

In [0]:
df = pd.json_normalize(data)
print(df)

In [0]:
df_exploded = df.explode("skills")
print(df_exploded)

In [0]:
response = {
  "company": "Infocepts",
  "location": "Mumbai",
  "departments": [
    {
      "dept_id": 10,
      "dept_name": "Sales",
      "employees": [
        {"emp_id": 1, "name": "John", "salary": 12000},
        {"emp_id": 2, "name": "Neha", "salary": 11000}
      ]
    },
    {
      "dept_id": 20,
      "dept_name": "IT",
      "employees": [
        {"emp_id": 3, "name": "Rohan", "salary": 15000}
      ]
    }
  ]
}

dept_df = pd.json_normalize(response, record_path=["departments"])
print(dept_df)

### 6. Flattening and Writing to Files
Once flattened, write data for downstream use:

In [0]:
dept_df.to_csv("/Volumes/thedataengineering_00/data/sampledata/sampleimages/department_flat.csv", index=False)

### 7. Parsing JSON from API Directly
You can combine this with your Day 18 learning.

In [0]:
import requests, pandas as pd

url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)
data = response.json()

# Flatten nested address and company details
df = pd.json_normalize(data)
print(df.columns)

### 8. Handling Large JSON Files (Performance Tip)
For huge JSON files (hundreds of MBs), read in chunks:

In [0]:
data_iter = pd.read_json("large_data.json", lines=True, chunksize=10000)
for chunk in data_iter:
    process_chunk(chunk)

# Summary

You've now mastered the art of converting complex JSON → tabular data - a vital step in any ingestion pipeline.

Practice Tasks

- Read a JSON file with nested address and flatten into columns.
-  Use .explode() to expand skills for each employee.
-  Flatten the "departments → employees" JSON structure into one table.
-  Fetch API data (from https://jsonplaceholder.typicode.com/users) and flatten it.
-  Save the final flattened table into a CSV for analysis.